In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [8]:
df = pd.read_csv("../../data/raw/Customers.csv")
df.head()


,CustomerID,Gender,Age,Annual Income ($),Spending Score (1-100),Profession,Work Experience,Family Size
0,1,Male,19,15000,39,Healthcare,1,4
1,2,Male,21,35000,81,Engineer,3,3
2,3,Female,20,86000,6,Engineer,1,1
3,4,Female,23,59000,77,Lawyer,0,2
4,5,Female,31,38000,40,Entertainment,2,6


In [10]:
#Remove CustomerID from dataframe since no predictive power.  Renamed column names to snake case for easier reference in code
df = df.rename(
    columns = {
        'Gender': 'gender',
        'Age': 'age',
        'Annual Income ($)': 'annual_income',
        'Spending Score (1-100)': 'spending_score',
        'Profession': 'profession',
        'Work Experience': 'work_experience',
        'Family Size': 'family_size',
    }
)

In [11]:
# define features and target
features = ["age","annual_income","work_experience","family_size"]
X = df[features].copy()
y = df["spending_score"].astype(float).values

In [12]:
# Train / test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
#Scale features

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

n_features = X_train_scaled.shape[1]

In [14]:
#Define Neural network model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Dense(64, activation='relu', input_shape=(n_features,)),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

c:\Users\saqib\UT_DSI_Project\predict_customer_spending_score\Project-env\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
# Compile
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

In [16]:
# Train
model.fit(
    X_train_scaled,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 3273.9585 - mae: 49.8901 - val_loss: 3315.4719 - val_mae: 50.3885
Epoch 2/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2935.2319 - mae: 46.5208 - val_loss: 2813.1528 - val_mae: 45.3668
Epoch 3/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2296.8882 - mae: 40.0650 - val_loss: 1981.2535 - val_mae: 36.6904
Epoch 4/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1470.9812 - mae: 31.3527 - val_loss: 1167.7817 - val_mae: 28.4508
Epoch 5/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 955.0537 - mae: 25.6798 - val_loss: 878.6887 - val_mae: 25.2375
Epoch 6/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 853.1320 - mae: 24.4288 - val_loss: 859.6130 - val_mae: 24.8599
Epoch 7/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 844.3885 - mae: 24.3061 - val_loss: 857.5653 - val_mae: 24.8431
Epoch 8/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 838.6351 - mae: 24.2365 - val_loss: 854.9106 - val_mae: 24.8181
Epoch 9/30
40/40

In [17]:
loss, mae = model.evaluate(X_test_scaled, y_test, verbose=0)
print("Test MSE (loss):", loss)
print("Test MAE:", mae)

Test MSE (loss): 816.1949462890625
Test MAE: 23.940332412719727


##This shows the model is not learning anything useful. Spending predictions are off by 24 points which is significant for a range 1 - 100.  Loss is very high too. These are the same results as the regressions experiments.